# CLEAR data preparation pipeline (PER-15)

This notebook turns the raw phone footage in the team Drive into a clean dataset that the model team can annotate and train on. Each numbered section is one stage. A stage reads what the previous one wrote, so after a crash you can rerun from that stage, and stages skip any work that is already done.

| # | Stage | What it produces |
|---|-------|------------------|
| 1 | Inventory | `inventory.csv` with each raw clip's codec, resolution, fps, length and recording time |
| 2 | Session metadata | `clip_metadata.csv`, where you fill in the court, camera position and lighting, plus a session ID for each clip |
| 3 | Compress and format | H.264 1080p 30 fps `.mp4` with no audio, plus `timestamps.npy` for each clip |
| 4 | Line detection and calibration | Reference frame, line mask and `calib.json` for each clip, using PER-6 anchor calibration |
| 5 | Frames for annotation | Sharp, de-duplicated JPEGs ready to upload to Roboflow |
| 6 | Augmentation | Lighting, weather and small camera-angle variations, with labels if they exist |
| 7 | Splits and manifest | train/val/test split by session, a `mini` subset and `manifest.csv` |
| 8 | Package | A zip of the processed dataset to upload to the Drive |

**Output layout** (modelled on [demalenk/caltennis](https://huggingface.co/datasets/demalenk/caltennis)):

```
clear_dataset/
  manifest.csv
  sessions/<MM_DD_YYYY_HH_MM_courtN>/<video_id>.mp4
  sessions/<MM_DD_YYYY_HH_MM_courtN>/<video_id>_timestamps.npy
  camera_calibration/<session>/<video_id>/calib.json   (+ reference.jpg, line_mask.png, preview.jpg)
  frames/<session>__<video_id>__t000123.jpg              (upload these to Roboflow)
  line_images/photo__<name>.jpg + masks/                 (photos for line annotation)
  augmented/...
  splits.json
```

## 0. Setup

Requirements: `ffmpeg` and `ffprobe` on PATH, plus `pip install opencv-python-headless numpy pandas albumentations scipy pillow pillow-heif`.

Run the notebook from the repo root, or from `notebooks/`. When `src/calibration` (PER-6) can be imported, stage 4 uses the real anchor calibration. When it can't, stage 4 still produces line masks and marks each clip `needs_taps`.

In [ ]:
import json, os, re, shutil, subprocess, sys, zipfile
from datetime import datetime, timedelta
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

if Path.cwd().name in ("notebooks", "data_pipeline"):  # paths below are relative to the repo root
    os.chdir("..")

# ---------------- CONFIG: edit these ----------------
RAW_DIR   = Path("data/raw")            # downloaded Drive folders (e.g. data/raw/Videos/Night/*.mov)
PHOTOS_DIR = Path("data/raw/Photos")    # still photos of the courts (optional)
OUT_DIR   = Path("data/clear_dataset")  # everything this notebook writes
TAPS_DIR  = Path("data/taps")           # taps/<session>/<video_id>.json, anchor name -> [u, v] (from PER-6)

TARGET_HEIGHT = 1080       # 1080p. Faint lines vanish at 720p (PER-6 finding), so do not go lower.
TARGET_FPS    = 30         # constant frame rate. Phones record variable fps, which breaks frame timing.
CRF           = 23         # H.264 quality: 18 = near-lossless, 23 = default, 28 = small
PRESET        = "medium"   # use "veryfast" for quick runs
DEFAULT_COURT = 1          # used when clip_metadata.csv does not say which court
SESSION_GAP_MIN = 45       # clips recorded within this many minutes of each other share a session

FRAME_EVERY_S   = 1.0      # sample one candidate frame per second
MIN_CHANGE      = 2.0      # mean grey-level change (0-255) vs recently kept frames; below = near-duplicate
BLUR_REL_MIN    = 0.5      # drop frames less than half as sharp as the clip's median frame
MAX_FRAMES_PER_CLIP = 150

AUG_PER_IMAGE = 3          # augmented copies per frame
SPLIT = {"train": 0.7, "val": 0.15, "test": 0.15}
MINI_PER_SESSION = 2       # clips per session in the 'mini' subset (like caltennis 'mini')
SEED = 42
# ----------------------------------------------------

VIDEO_EXT = {".mp4", ".mov", ".m4v", ".avi", ".mkv"}
PHOTO_EXT = {".jpg", ".jpeg", ".png", ".heic"}
for d in ["sessions", "camera_calibration", "frames", "augmented"]:
    (OUT_DIR / d).mkdir(parents=True, exist_ok=True)
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "Install ffmpeg first"

# Try to use the PER-6 calibration code from the repo.
for root in [Path.cwd(), Path.cwd().parent]:
    if (root / "src" / "calibration").exists() and str(root) not in sys.path:
        sys.path.insert(0, str(root))
try:
    from src.calibration.anchor_lines import calibrate_from_taps, line_mask
    HAVE_PER6 = True
except Exception as e:  # pragma: no cover
    HAVE_PER6 = False
    print("PER-6 calibration not importable, falling back to line masks only:", e)
print("PER-6 available:", HAVE_PER6)

In [ ]:
# Fallback line mask: the same idea as PER-6's line_mask (bright, low-saturation thin strokes on clay).
if not HAVE_PER6:
    def line_mask(frame_bgr, roi=None):
        h, w = frame_bgr.shape[:2]
        L = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)[:, :, 0]
        S = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)[:, :, 1]
        k = max(9, (w // 70) | 1)
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        score = cv2.morphologyEx(L, cv2.MORPH_TOPHAT, ker).astype(np.float32)
        score += cv2.morphologyEx(S, cv2.MORPH_BLACKHAT, ker).astype(np.float32)
        region = roi > 0 if roi is not None else np.ones((h, w), bool)
        vals = score[region]
        med = float(np.median(vals)); mad = float(np.median(np.abs(vals - med))) + 1e-6
        thresh = max(15.0, med + 8.0 * 1.4826 * mad)
        return (((score > thresh) & region).astype(np.uint8) * 255)

## 1. Inventory

**Why:** before changing anything, we record what we actually have. It also reads **which phone and which lens** recorded each clip. This matters because the ultrawide lens (13 mm equivalent) bends straight court lines into curves, and a homography assumes straight lines. Clips shot on the ultrawide lens are flagged, and for those we either correct lens distortion or re-record on the main lens. iPhone `.mov` files are usually HEVC with **variable** frame rate and a rotation flag, and Android `.mp4` files differ again. Knowing each clip's format tells us what stage 3 has to fix. The recording time (from the file metadata, or failing that the file name) is what lets us group clips into sessions.

In [ ]:
def ffprobe(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-print_format", "json", "-show_format", "-show_streams", str(path)],
                         capture_output=True, text=True, check=True).stdout
    return json.loads(out)

def _fps(s):
    num, den = (s or "0/1").split("/")
    return float(num) / float(den) if float(den) else 0.0

def _recorded_at(path, info):
    tags = {**info.get("format", {}).get("tags", {})}
    for st in info.get("streams", []):
        tags.update({k: v for k, v in st.get("tags", {}).items() if k not in tags})
    for key in ("com.apple.quicktime.creationdate", "creation_time"):
        if key in tags:
            try:
                ts = pd.Timestamp(tags[key])
                if ts.tzinfo is not None:  # store local (Kampala) time, naive
                    ts = ts.tz_convert("Africa/Kampala").tz_localize(None)
                return ts.to_pydatetime()
            except Exception:
                pass
    m = re.search(r"(20\d{6})_(\d{6})", path.name)  # Android style 20261021_195013.mp4
    if m:
        return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S")
    return datetime.fromtimestamp(path.stat().st_mtime)

def _rotation(v):
    for sd in v.get("side_data_list", []) or []:
        if "rotation" in sd:
            return int(sd["rotation"])
    return int(v.get("tags", {}).get("rotate", 0))

def _all_tags(info):
    tags = dict(info.get("format", {}).get("tags", {}))
    for st in info.get("streams", []):
        for k, v in st.get("tags", {}).items():
            tags.setdefault(k, v)
    return {k.lower(): v for k, v in tags.items()}

def camera_info(info):
    # Which phone and which lens recorded this clip, read from the file's own metadata.
    # iPhones store model, lens and 35mm-equivalent focal length; Android phones usually store only make/model.
    t = _all_tags(info)
    make = t.get("com.apple.quicktime.make") or t.get("com.android.manufacturer") or t.get("make", "")
    model = t.get("com.apple.quicktime.model") or t.get("com.android.model") or t.get("model", "")
    lens_model = t.get("com.apple.quicktime.camera.lens_model", "")
    f35 = t.get("com.apple.quicktime.camera.focal_length.35mm_equivalent", "")
    f35 = float(f35) if str(f35).replace(".", "", 1).isdigit() else None
    lens = ""
    if f35:
        lens = "ultrawide" if f35 < 18 else "main" if f35 < 50 else "telephoto"
    elif lens_model:
        m = re.search(r"([\d.]+)mm", lens_model)
        mm = float(m.group(1)) if m else None
        lens = ("ultrawide" if "ultra" in lens_model.lower() or (mm and mm < 2.5)
                else "telephoto" if "tele" in lens_model.lower() else "main")
    return dict(device_make=make, device_model=model, lens_model=lens_model, focal_35mm=f35, lens=lens)

rows = []
for p in sorted(RAW_DIR.rglob("*")):
    if p.suffix.lower() not in VIDEO_EXT or OUT_DIR in p.parents:
        continue
    info = ffprobe(p)
    v = next(s for s in info["streams"] if s["codec_type"] == "video")
    rows.append(dict(
        raw_path=str(p), file=p.name, folder=p.parent.name,
        codec=v.get("codec_name"), width=v.get("width"), height=v.get("height"), rotation=_rotation(v),
        avg_fps=round(_fps(v.get("avg_frame_rate")), 3), r_fps=round(_fps(v.get("r_frame_rate")), 3),
        duration_s=round(float(info["format"].get("duration", 0)), 2),
        size_mb=round(p.stat().st_size / 1e6, 1),
        has_audio=any(s["codec_type"] == "audio" for s in info["streams"]),
        recorded_at=_recorded_at(p, info),
        **camera_info(info),
    ))
inventory = pd.DataFrame(rows)
inventory["variable_fps"] = (inventory["avg_fps"] - inventory["r_fps"]).abs() > 0.5 if len(inventory) else []
inventory.to_csv(OUT_DIR / "inventory.csv", index=False)
uw = inventory[inventory["lens"] == "ultrawide"] if len(inventory) else []
if len(uw):
    print("WARNING: recorded on the ULTRAWIDE lens (bent lines, homography less accurate):", ", ".join(uw["file"]))
print(f"{len(inventory)} clips, {inventory['size_mb'].sum():.0f} MB, {inventory['duration_s'].sum()/60:.1f} min")
inventory

## 2. Session metadata

**Why:** caltennis groups clips by **session**, meaning one recording sitting on one court. We do the same, because the train/test split in stage 7 must keep whole sessions together. If two clips from the same afternoon land in train and test, the test score looks better than it really is.

The first run writes `clip_metadata.csv`, with the court, camera position and lighting guessed from the folder name (`Night` means night). **Open it, correct anything wrong, and rerun this cell.** Your edits are kept.

**Cutting out unneeded parts (warm-ups, walking to the camera, breaks):** don't edit the videos in a video editor. Editors re-encode the file and usually strip the recording time, which this pipeline needs. Instead, put the parts to **keep** in the `keep_segments` column, for example `0:10-1:35; 2:40-4:05` (minutes:seconds). Stage 3 cuts exactly those parts from the untouched original. Each kept part becomes its own clip (`..._s1`, `..._s2`). This makes the cuts reproducible: anyone can see what was removed and change it later. Leave the column empty to keep the whole clip, and set `include` to False to drop a clip entirely.

Session ID format: `MM_DD_YYYY_HH_MM_court<N>`, where the time is the start of the first clip in the session. Video ID format: `<camera_position>_<HHMMSS>`, for example `near_left_194859`.

In [ ]:
meta_path = OUT_DIR / "clip_metadata.csv"
auto = pd.DataFrame({
    "file": inventory["file"],
    "court": DEFAULT_COURT,
    "camera_position": "near",   # near / far / near_left / near_right / overhead ...
    "lighting": np.where(inventory["folder"].str.lower().str.contains("night"), "night", "day"),
    "include": True,
    "keep_segments": "",         # e.g. "0:10-1:35; 2:40-4:05" (min:sec). Empty = keep the whole clip.
    "notes": "",
})
if meta_path.exists():
    user = pd.read_csv(meta_path, keep_default_na=False)
    for col in auto.columns:  # older metadata files may lack new columns
        if col not in user.columns: user[col] = auto[col].iloc[0] if col != "file" else ""
    meta = pd.concat([user, auto[~auto["file"].isin(user["file"])]], ignore_index=True)
else:
    meta = auto
meta.to_csv(meta_path, index=False)

# Measured values from the inventory win over same-named columns typed into the metadata file;
# a typed value is used only where the file itself said nothing (e.g. lens on Android clips).
overlap = [c for c in meta.columns if c in inventory.columns and c != "file"]
clips = inventory.merge(meta, on="file", suffixes=("", "__typed"))
for c in overlap:
    measured = clips[c].replace("", np.nan)
    clips[c] = measured.where(measured.notna(), clips[c + "__typed"].replace("", np.nan))
    clips = clips.drop(columns=c + "__typed")
# Blank cells (not filled in yet) fall back to safe defaults.
clips["court"] = pd.to_numeric(clips["court"], errors="coerce").fillna(DEFAULT_COURT).astype(int)
clips["camera_position"] = clips["camera_position"].replace("", "cam").fillna("cam")
clips["lighting"] = clips["lighting"].replace("", "day").fillna("day")
clips["include"] = clips["include"].astype(str).str.lower().isin(["true", "1", "yes"])
clips = clips[clips["include"]].sort_values(["court", "recorded_at"]).reset_index(drop=True)

def _secs(t):
    parts = [float(x) for x in str(t).strip().split(":")]
    return sum(v * 60 ** i for i, v in enumerate(reversed(parts)))

def parse_segments(spec, duration):
    # "0:10-1:35; 2:40-4:05" -> [(10, 95), (160, 245)]. Empty -> whole clip.
    spec = str(spec or "").strip()
    if not spec or spec.lower() == "nan":
        return [(0.0, float(duration))]
    segs = []
    for part in re.split(r"[;,]", spec):
        if "-" not in part: continue
        a, b = part.split("-", 1)
        a, b = _secs(a), min(_secs(b), float(duration))
        assert b > a, f"bad segment {part!r}"
        segs.append((a, b))
    return segs

# One row per kept segment. A clip with 2 segments becomes 2 output clips (..._s1, ..._s2).
rows = []
for _, r in clips.iterrows():
    segs = parse_segments(r.get("keep_segments", ""), r.duration_s)
    for k, (a, b) in enumerate(segs, 1):
        rr = r.copy(); rr["seg_start_s"], rr["seg_end_s"] = a, b
        rr["seg_suffix"] = f"_s{k}" if len(segs) > 1 or a > 0 or b < r.duration_s - 0.05 else ""
        rr["recorded_at"] = r.recorded_at + timedelta(seconds=a)
        rr["duration_s"] = round(b - a, 2)
        rows.append(rr)
clips = pd.DataFrame(rows).reset_index(drop=True)

# New session whenever the court changes or there is a gap longer than SESSION_GAP_MIN.
session_ids, start = [], None
for i, r in clips.iterrows():
    prev = clips.iloc[i - 1] if i else None
    if prev is None or r.court != prev.court or \
       r.recorded_at - (prev.recorded_at + timedelta(seconds=prev.duration_s)) > timedelta(minutes=SESSION_GAP_MIN):
        start = r.recorded_at
    session_ids.append(f"{start:%m_%d_%Y_%H_%M}_court{int(r.court)}")
clips["session_id"] = session_ids
clips["video_id"] = [f"{c}_{(t - timedelta(seconds=a)):%H%M%S}{sfx}"
                     for c, t, a, sfx in zip(clips["camera_position"], clips["recorded_at"], clips["seg_start_s"], clips["seg_suffix"])]
dup = clips.duplicated(["session_id", "video_id"], keep=False)
clips.loc[dup, "video_id"] += "_" + clips.loc[dup].groupby(["session_id", "video_id"]).cumcount().astype(str)
clips[["file", "session_id", "video_id", "lighting", "seg_start_s", "seg_end_s", "duration_s"]]

## 3. Compress and format

**Why each setting:**
- **H.264 `.mp4`**: every tool (OpenCV, Roboflow, PyTorch decoders, browsers) can read it. HEVC `.mov` often fails silently in OpenCV on Linux servers.
- **1080p**: PER-6 found that faint lines disappear at lower resolution, so this is the floor. Phones recording 4K become roughly 4 to 8 times smaller.
- **Constant 30 fps** (`fps=30`, `-fps_mode cfr`): phones record variable frame rate, so "frame 300" is not always 10.000 s in. A constant rate makes frame number and time agree, which matters for ball speed and bounce timing.
- **Rotation applied**: ffmpeg applies the phone's rotation flag, so frames come out upright and the flag is removed.
- **No audio**: smaller files. (If anyone later wants racket-hit sounds for timing, set `KEEP_AUDIO = True`.)
- **`+faststart`**: lets the video stream from the Drive or a browser before it has fully downloaded.

`timestamps.npy` holds each frame's time in seconds from the start of the clip (float64). The absolute start time is `recorded_at` in the manifest.

In [ ]:
KEEP_AUDIO = False

def compress(src, dst, start=0.0, end=None):
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(".part.mp4")
    cut = ["-ss", f"{start:.3f}"] + (["-to", f"{end:.3f}"] if end else [])  # we re-encode, so cuts are frame-accurate
    cmd = ["ffmpeg", "-y", "-v", "error"] + cut + ["-i", str(src),
           "-vf", f"scale=-2:'min({TARGET_HEIGHT},ih)':flags=lanczos,fps={TARGET_FPS}",
           "-fps_mode", "cfr", "-c:v", "libx264", "-preset", PRESET, "-crf", str(CRF),
           "-pix_fmt", "yuv420p", "-movflags", "+faststart"]
    cmd += ["-c:a", "aac", "-b:a", "96k"] if KEEP_AUDIO else ["-an"]
    subprocess.run(cmd + [str(tmp)], check=True)
    tmp.rename(dst)

def frame_times(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "packet=pts_time",
                          "-of", "csv=p=0", str(path)], capture_output=True, text=True, check=True).stdout
    return np.sort(np.array([float(x) for x in out.split() if x and x != "N/A"], dtype=np.float64))

out_paths = []
for _, r in clips.iterrows():
    dst = OUT_DIR / "sessions" / r.session_id / f"{r.video_id}.mp4"
    if not dst.exists():
        print("compressing", r.file, "->", dst.relative_to(OUT_DIR))
        compress(Path(r.raw_path), dst, r.seg_start_s, r.seg_end_s)
    ts_path = dst.with_name(f"{r.video_id}_timestamps.npy")
    if not ts_path.exists():
        ts = frame_times(dst); np.save(ts_path, ts - ts[0])
    out_paths.append(dst)
clips["video"] = [str(p.relative_to(OUT_DIR)) for p in out_paths]
clips["timestamps"] = [str(p.with_name(p.stem + "_timestamps.npy").relative_to(OUT_DIR)) for p in out_paths]
clips["out_mb"] = [round(p.stat().st_size / 1e6, 1) for p in out_paths]
clips["n_frames"] = [len(np.load(OUT_DIR / t)) for t in clips["timestamps"]]
raw_mb_total = clips.drop_duplicates("file")["size_mb"].sum()  # a cut clip's source is counted once
print(f"raw {raw_mb_total:.0f} MB -> processed {clips['out_mb'].sum():.0f} MB "
      f"({100 * (1 - clips['out_mb'].sum() / max(raw_mb_total, 1e-9)):.0f}% smaller)")
clips[["video_id", "codec", "width", "height", "avg_fps", "size_mb", "out_mb", "n_frames"]]

## 4. Line detection and calibration (PER-6)

**Why:** each clip needs a `calib.json`, the homography from court metres to image pixels, like caltennis. On our practice courts the paint is faint or gone, so PER-6 calibrates from **tapped anchors** (net post bases, plus any surviving line corners) and then tightens the fit with whatever line pixels are still visible.

For each clip this stage:
1. Saves a sharp **reference frame** (`reference.jpg`, at the processed 1080p size). **Tap anchors on this image**, not the raw 4K video, so the pixel coordinates match.
2. Saves the **line mask** (`line_mask.png`): bright, grey, thin strokes on the clay. It's a quick check of how much paint is visible.
3. If `TAPS_DIR/<session>/<video_id>.json` exists, runs `calibrate_from_taps` and writes `calib.json` and `preview.jpg`. If it doesn't, it writes a `calib.json` with `status: needs_taps`.

Because the camera is fixed for a whole clip, **one calibration per clip** is enough. If the phone is bumped mid-clip, PER-6's `track_homography` handles it at run time.

**Photos (for lines):** every photo in `PHOTOS_DIR` is resized to 1080p and copied to `line_images/`, with its automatic line mask in `line_images/masks/` and an overlay in `camera_calibration/_photos/` for checking. Photos are better than video frames for lines: they're sharper (no motion blur or video compression), so faint paint survives. Upload `line_images/` to a **separate** Roboflow project for line annotation, to keep it apart from the ball and player frames.

In [ ]:
def sharpest_frame(video, n_probe=15):
    cap = cv2.VideoCapture(str(video)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    best, best_score = None, -1
    for idx in np.linspace(0, max(n - 1, 0), n_probe).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx)); ok, f = cap.read()
        if ok:
            s = cv2.Laplacian(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            if s > best_score: best, best_score = f, s
    cap.release()
    return best

def overlay_mask(frame, mask):
    out = frame.copy(); out[mask > 0] = (0, 255, 255)
    return cv2.addWeighted(frame, 0.5, out, 0.5, 0)

calib_status = []
for _, r in clips.iterrows():
    cdir = OUT_DIR / "camera_calibration" / r.session_id / r.video_id
    cdir.mkdir(parents=True, exist_ok=True)
    frame = sharpest_frame(OUT_DIR / r.video)
    cv2.imwrite(str(cdir / "reference.jpg"), frame)
    mask = line_mask(frame)
    cv2.imwrite(str(cdir / "line_mask.png"), mask)
    cv2.imwrite(str(cdir / "line_overlay.jpg"), overlay_mask(frame, mask))
    line_px_pct = round(100 * float((mask > 0).mean()), 3)

    taps_file = TAPS_DIR / r.session_id / f"{r.video_id}.json"
    calib = {"session_id": r.session_id, "video_id": r.video_id, "image_size": [frame.shape[1], frame.shape[0]],
             "line_pixels_pct": line_px_pct}
    if HAVE_PER6 and taps_file.exists():
        taps = {k: (float(v[0]), float(v[1])) for k, v in json.loads(taps_file.read_text()).items()}
        cal, report = calibrate_from_taps(taps, frame)
        cal.save(cdir / "calib_per6.json")
        calib.update(json.loads((cdir / "calib_per6.json").read_text()))
        calib.update({"status": report.get("status"), "report": report})
        try:
            from src.calibration.virtual_court import VirtualCourtRenderer
            cv2.imwrite(str(cdir / "preview.jpg"), VirtualCourtRenderer(calibrator=cal, court_mode="doubles").draw(frame, draw_anchors=True))
        except Exception as e:
            print("preview skipped:", e)
    else:
        calib["status"] = "needs_taps"
        calib["taps_expected_at"] = str(taps_file)
    (cdir / "calib.json").write_text(json.dumps(calib, indent=2, default=str))
    calib_status.append(calib["status"])
clips["calibration"] = [f"camera_calibration/{s}/{v}/calib.json" for s, v in zip(clips.session_id, clips.video_id)]
clips["calib_status"] = calib_status

# Still photos: these are mainly for LINE detection / line annotation.
# Each photo is resized to 1080p and saved to line_images/ (upload these to a separate Roboflow project for lines),
# together with its automatic line mask in line_images/masks/. The mask is a starting point: you can
# check it, and upload it as a pre-label so you only fix mistakes instead of tracing every line by hand.
line_dir = OUT_DIR / "line_images"; (line_dir / "masks").mkdir(parents=True, exist_ok=True)
photo_dir = OUT_DIR / "camera_calibration" / "_photos"
photos = [p for p in PHOTOS_DIR.rglob("*") if p.suffix.lower() in PHOTO_EXT] if PHOTOS_DIR.exists() else []
def read_image(p):
    if p.suffix.lower() == ".heic":  # iPhone/Samsung photos; OpenCV can't read HEIC
        try:
            from PIL import Image; import pillow_heif; pillow_heif.register_heif_opener()
            return cv2.cvtColor(np.array(Image.open(p).convert("RGB")), cv2.COLOR_RGB2BGR)
        except ImportError:
            print("pip install pillow pillow-heif to read HEIC photos"); return None
    return cv2.imread(str(p))

for p in photos:
    img = read_image(p)
    if img is None:
        print("cannot read", p.name, "(convert HEIC to JPG first)"); continue
    if img.shape[0] > TARGET_HEIGHT:
        img = cv2.resize(img, (round(img.shape[1] * TARGET_HEIGHT / img.shape[0]), TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
    photo_dir.mkdir(parents=True, exist_ok=True)
    m = line_mask(img)
    cv2.imwrite(str(line_dir / f"photo__{p.stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    cv2.imwrite(str(line_dir / "masks" / f"photo__{p.stem}.png"), m)
    cv2.imwrite(str(photo_dir / f"{p.stem}_line_overlay.jpg"), overlay_mask(img, m))
print(clips["calib_status"].value_counts().to_dict(), f"| {len(photos)} photos processed")

## 5. Frames for annotation (Roboflow)

**Why:** we annotate **images**, not videos, and consecutive video frames are almost identical, so labelling all of them wastes hours and adds nothing. This stage:
- takes one candidate every `FRAME_EVERY_S` seconds,
- drops **blurry** frames, since they're hard to label and teach the model little. Sharpness (variance of the Laplacian) is compared with **the clip's own median**, not a fixed number, because night clips are darker and would otherwise be thrown out entirely,
- drops **near-duplicates**: a frame is kept only if it differs from the recently kept frames by at least `MIN_CHANGE` grey levels on average (computed on a small 96x54 thumbnail). The camera is fixed, so "different" here means the players or ball have moved. An empty court doesn't produce 100 copies of the same picture.

File names carry the session, clip and time, so after the Roboflow export every label can be traced back to its video and `calib.json`.

**Upload to Roboflow:** drag the `frames/` folder into a Roboflow project. Keep the file names.

In [ ]:
def thumb(img):
    return cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (96, 54), interpolation=cv2.INTER_AREA).astype(np.float32)

def sharpness(img):
    return cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()

frame_rows = []
for _, r in clips.iterrows():
    cap = cv2.VideoCapture(str(OUT_DIR / r.video))
    step = max(1, int(round(TARGET_FPS * FRAME_EVERY_S)))
    # Pass 1: read the candidate frames and their sharpness.
    cands, idx = [], 0
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, f = cap.read()
        if not ok: break
        cands.append((idx, sharpness(f))); idx += step
    med_sharp = np.median([s for _, s in cands]) if cands else 0
    # Pass 2: keep sharp frames that differ from what we already kept.
    kept, recent = 0, []
    for idx, blur in cands:
        if kept >= MAX_FRAMES_PER_CLIP: break
        if blur < BLUR_REL_MIN * med_sharp: continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, f = cap.read()
        if not ok: continue
        th = thumb(f)
        if all(np.abs(th - o).mean() >= MIN_CHANGE for o in recent[-30:]):
            t = idx / TARGET_FPS
            name = f"{r.session_id}__{r.video_id}__t{int(t * 1000):07d}.jpg"
            dst = OUT_DIR / "frames" / name
            if not dst.exists(): cv2.imwrite(str(dst), f, [cv2.IMWRITE_JPEG_QUALITY, 92])
            frame_rows.append(dict(image=f"frames/{name}", session_id=r.session_id, video_id=r.video_id,
                                   frame_idx=idx, t_s=round(t, 3), blur=round(blur, 1), lighting=r.lighting))
            recent.append(th); kept += 1
    cap.release()
frames = pd.DataFrame(frame_rows)
frames.to_csv(OUT_DIR / "frames.csv", index=False)
print(f"{len(frames)} frames kept for annotation")
frames.groupby("video_id").size() if len(frames) else frames

## 6. Augmentation

**Why:** we have only a few courts, a few phones and a few lighting conditions. Augmentation creates realistic variations so the model doesn't overfit to them:
- **Lighting**: brightness and contrast, gamma, and colour shifts for evening light and different phones. A night-style darkening is applied to day frames.
- **Weather and camera quality**: motion blur, Gaussian noise, JPEG artifacts and rain streaks.
- **Camera placement**: *small* rotation, scale and perspective changes, for a slightly different tripod position. Kept small so the court geometry stays realistic.
- **No horizontal flip by default.** Flipping swaps left and right, so a `net_post_left` keypoint would end up on the right post. Only turn it on for labels that don't depend on side (for example the ball).

**Important: augment AFTER annotation.** Augmenting unlabelled frames would mean labelling every copy by hand. The proper flow is:
1. Upload `frames/` to Roboflow and annotate.
2. Export from Roboflow in **YOLO** format, into `ROBOFLOW_EXPORT` below (`images/` + `labels/`).
3. Rerun this cell. Bounding boxes and keypoints are moved together with the image.

If there's no export yet, it augments the raw frames (images only), so you can preview the variations and use them for line detection robustness tests.

In [ ]:
import albumentations as A

ROBOFLOW_EXPORT = Path("data/roboflow_export/train")  # contains images/ and labels/ (YOLO format)
ALLOW_HFLIP = False

def build_aug(with_boxes, with_kps):
    t = [
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.RandomGamma(gamma_limit=(60, 140), p=0.4),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.4),
        A.OneOf([A.MotionBlur(blur_limit=(3, 9)), A.GaussianBlur(blur_limit=(3, 5))], p=0.3),
        A.GaussNoise(std_range=(0.02, 0.08), p=0.3),
        A.ImageCompression(quality_range=(40, 90), p=0.3),
        A.RandomRain(drop_length=15, blur_value=3, brightness_coefficient=0.85, p=0.1),
        A.Affine(scale=(0.92, 1.08), translate_percent=(-0.03, 0.03), rotate=(-3, 3), p=0.5),
        A.Perspective(scale=(0.01, 0.04), p=0.3),
    ]
    if ALLOW_HFLIP:
        t.append(A.HorizontalFlip(p=0.5))
    kw = {}
    if with_boxes: kw["bbox_params"] = A.BboxParams(format="yolo", label_fields=["box_cls"], min_visibility=0.3)
    if with_kps:   kw["keypoint_params"] = A.KeypointParams(format="xy", remove_invisible=False)
    return A.Compose(t, seed=SEED, **kw)

def night_style(img):
    # Darken, cool the colour and add sensor noise: a cheap stand-in for floodlit evening footage.
    out = img.astype(np.float32) * np.array([1.05, 0.8, 0.65]) * 0.45
    out += np.random.default_rng(SEED).normal(0, 6, img.shape)
    return np.clip(out, 0, 255).astype(np.uint8)

def read_yolo(label_path, w, h):
    # YOLO det: cls cx cy bw bh | YOLO pose: cls cx cy bw bh x1 y1 v1 x2 y2 v2 ...
    boxes, cls, kps, vis = [], [], [], []
    for line in label_path.read_text().split("\n"):
        v = line.split()
        if len(v) < 5: continue
        cls.append(int(v[0])); boxes.append([float(x) for x in v[1:5]])
        rest = [float(x) for x in v[5:]]
        for j in range(0, len(rest) - 2, 3):
            kps.append((rest[j] * w, rest[j + 1] * h)); vis.append(rest[j + 2])
    return boxes, cls, kps, vis

def write_yolo(path, boxes, cls, kps, vis, w, h, n_kp):
    lines = []
    for i, (b, c) in enumerate(zip(boxes, cls)):
        parts = [str(int(c))] + [f"{x:.6f}" for x in b]
        for j in range(n_kp):
            x, y = kps[i * n_kp + j]; v = vis[i * n_kp + j]
            if not (0 <= x < w and 0 <= y < h): v = 0
            parts += [f"{x / w:.6f}", f"{y / h:.6f}", f"{int(v)}"]
        lines.append(" ".join(parts))
    path.write_text("\n".join(lines))

frame_lighting = {Path(i).name: l for i, l in zip(frames.get("image", []), frames.get("lighting", []))}
aug_dir = OUT_DIR / "augmented"
labelled = (ROBOFLOW_EXPORT / "images").exists()
src_images = sorted((ROBOFLOW_EXPORT / "images").glob("*")) if labelled else sorted((OUT_DIR / "frames").glob("*.jpg"))
(aug_dir / "images").mkdir(parents=True, exist_ok=True); (aug_dir / "labels").mkdir(exist_ok=True)
n_written = 0
for p in src_images:
    img = cv2.imread(str(p))
    if img is None: continue
    h, w = img.shape[:2]
    boxes, cls, kps, vis = ([], [], [], [])
    lp = ROBOFLOW_EXPORT / "labels" / f"{p.stem}.txt"
    if labelled and lp.exists():
        boxes, cls, kps, vis = read_yolo(lp, w, h)
    n_kp = len(kps) // max(len(boxes), 1)
    aug = build_aug(bool(boxes), bool(kps))
    for k in range(AUG_PER_IMAGE):
        # One copy of each daytime frame gets the night look (only before annotation; labelled night data should be real).
        base = night_style(img) if (k == 0 and not labelled and frame_lighting.get(p.name) == "day") else img
        args = {"image": base}
        if boxes: args.update(bboxes=boxes, box_cls=cls)
        if kps: args["keypoints"] = kps
        res = aug(**args)
        name = f"{p.stem}__aug{k}"
        cv2.imwrite(str(aug_dir / "images" / f"{name}.jpg"), res["image"])
        if boxes:
            # Albumentations may drop boxes that leave the frame; keep keypoints only for surviving boxes.
            if len(res["bboxes"]) == len(boxes):
                write_yolo(aug_dir / "labels" / f"{name}.txt", res["bboxes"], res["box_cls"],
                           res.get("keypoints", []), vis, w, h, n_kp)
            else:
                (aug_dir / "images" / f"{name}.jpg").unlink()
                continue
        n_written += 1
print(f"{'labelled' if labelled else 'unlabelled (preview)'} augmentation: {n_written} images from {len(src_images)} sources")

## 7. Splits and manifest

**Why split by session:** frames from the same session look almost identical (same court, same light, same camera). If they're spread across train and test, the model gets tested on near-copies of its training data and the score is inflated. So whole sessions go into one split. With fewer than 3 sessions that isn't possible, and the split falls back to whole **clips** (it prints a warning).

`mini` is a small subset (a couple of clips per session) for quick debugging runs, like the caltennis `mini` split.

In [ ]:
rng = np.random.default_rng(SEED)
# Fallback unit is the source file, so two cut segments of one recording never land in different splits.
unit = "session_id" if clips["session_id"].nunique() >= 3 else "file"
if unit == "file":
    print("WARNING: fewer than 3 sessions, so splitting by clip. Record more sessions for an honest test set.")
groups = clips[unit].unique().tolist(); rng.shuffle(groups)
n = len(groups); n_tr = max(1, round(n * SPLIT["train"])); n_va = max(0, round(n * SPLIT["val"])) if n > 2 else 0
assign = {g: ("train" if i < n_tr else "val" if i < n_tr + n_va else "test") for i, g in enumerate(groups)}
if n >= 2 and "test" not in assign.values():
    assign[groups[-1]] = "test"
clips["split"] = clips[unit].map(assign)
clips["mini"] = clips.groupby("session_id").cumcount() < MINI_PER_SESSION
if len(frames):
    frames["split"] = frames["video_id"].map(dict(zip(clips.video_id, clips.split)))

manifest_cols = ["session_id", "video_id", "video", "seg_start_s", "seg_end_s", "timestamps", "calibration", "split", "mini", "court",
                 "camera_position", "lighting", "recorded_at", "duration_s", "n_frames", "calib_status",
                 "file", "codec", "width", "height", "rotation", "avg_fps", "variable_fps", "size_mb", "out_mb"]
# Any extra metadata columns you added (device, weather, players, ...) are carried into the manifest too.
extra_cols = [c for c in meta.columns if c in clips.columns and c not in manifest_cols and c not in ("include", "keep_segments")]
clips[manifest_cols + extra_cols].to_csv(OUT_DIR / "manifest.csv", index=False)
if len(frames): frames.to_csv(OUT_DIR / "frames.csv", index=False)
splits = {s: clips.loc[clips.split == s, "video"].tolist() for s in ["train", "val", "test"]}
splits["mini"] = clips.loc[clips.mini, "video"].tolist()
(OUT_DIR / "splits.json").write_text(json.dumps(splits, indent=2))
clips[["session_id", "video_id", "lighting", "split", "mini", "calib_status"]]

## 8. Package and summary

Zips the processed dataset (the raw files are not included) to upload to the team Drive, under `Processed/` next to `Videos/` and `Photos/`. Set `INCLUDE_AUGMENTED = False` to keep the zip small.

In [ ]:
INCLUDE_AUGMENTED = True
zip_path = OUT_DIR.parent / f"clear_dataset_{datetime.now():%Y%m%d}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:  # mp4/jpg are already compressed
    for p in OUT_DIR.rglob("*"):
        if p.is_file() and (INCLUDE_AUGMENTED or "augmented" not in p.parts) and not p.name.endswith(".part.mp4"):
            z.write(p, p.relative_to(OUT_DIR.parent))
summary = {
    "clips": len(clips), "sessions": clips.session_id.nunique(),
    "minutes": round(float(clips.duration_s.sum()) / 60, 1),
    "raw_mb": round(float(clips.drop_duplicates("file").size_mb.sum()), 1), "processed_mb": round(float(clips.out_mb.sum()), 1),
    "frames_for_annotation": len(frames), "augmented_images": n_written,
    "calibration": clips.calib_status.value_counts().to_dict(),
    "splits": clips.split.value_counts().to_dict(),
    "zip": str(zip_path), "zip_mb": round(zip_path.stat().st_size / 1e6, 1),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
summary